In [1]:
import os
from pathlib import Path
import sys
import numpy as np
import scvelo as scv

In [2]:
notebook_dir = os.getcwd()
notebook_dir

'c:\\Users\\OmerCagatayTalikaci\\Desktop\\github\\STVelo\\notebooks\\analysis\\nuc_cyto_vs_spl_uns'

In [3]:
Path(notebook_dir).parents[3]

WindowsPath('c:/Users/OmerCagatayTalikaci/Desktop/github')

In [4]:
os.chdir(Path(notebook_dir).parents[3])

In [5]:
stvelo_path = os.path.join(os.getcwd(),'STVelo','stvelo')

In [6]:
import sys  
sys.path.insert(1,stvelo_path)
from pipelines.simulation_3ode_class import Simulation3ODE
from pipelines.metrics import *
from pipelines.preprocessing import Preprocessing
from pipelines.compute_velocity import Velocities

In [7]:
# Order of correlations: [rho_alpha_beta, rho_alpha_nu, rho_alpha_gamma, rho_beta_nu, rho_beta_gamma, rho_nu_gamma]
# Order of sigmas : [sigma_log_alpha, sigma_log_beta, sigma_log_nu, sigma_log_gamma]
# Order of means : [alpha,beta,nu,gamma]

config_simulation = {'parameters':{ 'n_obs':800,
    'n_vars':300,
    'alpha':None,
    'beta':None,
    'nu' :None,
    'gamma':None,
    'alpha_':0,
    't_max':20,
    'noise_model':"normal",
    'noise_level':2,
    }, 

    'options': {'generate_parameters':True,
                'generate_switch_times':True,
                  'save':False,
                  'saving_path':None
                  },
    'coeff_generate_options':{ 
    'corr':[0.3, 0.2, -0.4, 0.3, -0.1, -0.2],
    'sd':[0.6, 0.6, 0.6, 0.6],
    'mean':[5, 1.8, 0.6, 0.3]} }


config_preprocessing = {'preprocess_params': {
     'min_counts': 200, 
     'min_cells': 5,
     'n_neighbors': 23,
     'n_pcs': 0,
     'min_dist': 1},
'functions_to_apply': {'filter_cells': True, 
                       'filter_genes': True, 
                       'normalize_total': True,
                       'log1p': True,
                       'pca': True,
                       'neighbors': True,
                       'umap': True,
                       'leiden': True,
                       'moments': True}}


# config_velocity = { 'velocity_types': ['dynamical','deterministic','stochastic','velovi']}
config_velocity = { 'velocity_types': ['deterministic']}



config_plotting = {
    'colorsets': ['leiden', 'clusters'],
    'velocity_embedding_stream': True,
    'velocity_embedding_grid': True,
    'velocity_embedding': True,
    'rank_velocity_genes': True,
    'velocity_confidence': True,
    'velocity_length': True
}



# Simulation 

In [27]:
simulator = Simulation3ODE(config = config_simulation) 

parameters generated with the covariance matrix: [[ 0.36   0.108  0.072 -0.144]
 [ 0.108  0.36   0.108 -0.036]
 [ 0.072  0.108  0.36  -0.072]
 [-0.144 -0.036 -0.072  0.36 ]]


In [12]:
adata_dict = simulator.simulate_multi_obs_var(n_obs_list=[800,1000], n_vars_list=[300])

In [28]:
adata_dict = simulator.simulation()

In [30]:
simulator.alpha

[224.20505977107157,
 111.21456612907178,
 164.0046592589489,
 220.06151618044638,
 187.8416694362139,
 96.84402643831955,
 85.8561914124583,
 177.07489326553437,
 241.4882298868233,
 122.14389810317853,
 169.616335613729,
 263.8074728587498,
 134.77142767716376,
 60.27886159427222,
 109.09318553219035,
 213.55930942895577,
 54.01853422441805,
 294.0756648985861,
 85.0117548104856,
 84.46124987913541,
 293.9440599839193,
 186.66171601629514,
 247.7054399218903,
 71.95060626669422,
 155.5279675642479,
 149.49360800368183,
 229.14334800331505,
 64.18927806332655,
 223.66890407002674,
 127.95640896826514,
 74.74294651175069,
 231.7323348536386,
 197.7749760681199,
 125.46895173568075,
 119.03391645211039,
 155.89461635999382,
 54.55243915519126,
 87.28636602530331,
 300.181032955702,
 86.04594281868948,
 263.47955647984816,
 67.28405428666642,
 56.90887379836002,
 161.1625018573869,
 158.9036463233148,
 346.594409177394,
 156.1437673874425,
 317.4928230239499,
 168.9810862702346,
 57.6633

In [31]:
adata = scv.datasets.simulation(800, alpha=simulator.alpha, beta=simulator.nu, gamma=simulator.gamma, n_vars=300 )

In [33]:
adata_dict['control'] = adata

# Preprocessing 

In [35]:
for key, adata in adata_dict.items():
    print(f'{key} is being preprocessed.')
    preprocessor = Preprocessing(adata,config_preprocessing)
    adata = preprocessor.preprocess_data()

adata_s_u_800obs_300genes is being preprocessed.
computing moments based on connectivities
    finished (0:00:00) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
adata_n_c_800obs_300genes is being preprocessed.
computing moments based on connectivities
    finished (0:00:00) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
control is being preprocessed.
computing moments based on connectivities
    finished (0:00:00) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)


# Velocity 

In [36]:
velocity_computer = Velocities(adata_dict,config_velocity)

Using device: cuda


In [37]:
adata_dict_velocity = velocity_computer.compute_velocities()

computing velocities
    finished (0:00:00) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)
computing velocity graph (using 1/24 cores)
or disable the progress bar using `show_progress_bar=False`.
    finished (0:00:00) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)
computing velocities
    finished (0:00:00) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)
computing velocity graph (using 1/24 cores)
    finished (0:00:00) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)
computing velocities
    finished (0:00:00) --> added 
    'velocity', velocity vectors for each individual cell (adata.layers)
computing velocity graph (using 1/24 cores)
    finished (0:00:00) --> added 
    'velocity_graph', sparse matrix with cosine correlations (adata.uns)


# Save adatas 

In [15]:
os.getcwd()

'c:\\Users\\OmerCagatayTalikaci\\Desktop\\github'

In [16]:
saving_path = os.path.join(os.getcwd(),'data','simulated')

In [17]:
for d in adata_dict_velocity.keys():
    print(d)
    adata_dict_velocity[d].write(os.path.join(saving_path,d+'.h5ad'))

adata_s_u_800obs_300genes_deterministic
adata_n_c_800obs_300genes_deterministic
adata_s_u_1000obs_300genes_deterministic
adata_n_c_1000obs_300genes_deterministic


# True Velocity

In [25]:
adata_n_c = adata_dict_velocity['adata_n_c_800obs_300genes_deterministic']

In [26]:
adata_n_c.layers

Layers with keys: unspliced, spliced, spliced_nuc, Ms, Mu, velocity

In [35]:
# Assuming you have calculated beta1
S_n = adata_n_c.layers['spliced_nuc']      # Spliced cytoplasmic counts S_c = s_c(t)
gamma = adata.var['true_gamma'].values  # Degradation rates per gene
nu = adata.var['true_nu'].values
S_c = adata.layers['spliced']
U = adata.layers['unspliced']



In [28]:
true_velocity = (nu * S_n) - (gamma * S_c)

In [29]:
true_velocity

array([[ -0.6886876 ,  35.6694559 , -39.10601073, ..., -26.31766187,
          0.        ,  11.76102436],
       [ -1.34039119,   0.        ,   0.        , ...,  29.01913401,
         37.85522784, -12.64969027],
       [-12.94874014, -70.27104651,  14.48156356, ...,  54.33644087,
          0.        ,   0.        ],
       ...,
       [-42.10286104,  47.35593756,   0.        , ...,   0.        ,
         29.66692119,  -1.70317075],
       [-42.29564386,   7.91767581,  -6.115035  , ...,   0.        ,
        104.42627531,  -7.37109628],
       [ -6.23440896, -68.85425444,   0.        , ...,  31.97018235,
          1.06189811,  14.6353845 ]])

In [36]:
true_velocity_2 = nu* U - gamma*S_c

In [32]:
velocity = adata_n_c.layers['velocity']

In [50]:
from sklearn.metrics.pairwise import cosine_similarity

In [48]:
from sklearn.metrics.pairwise import cosine_similarity
def compute_cosine_similarity(true_velocities, estimated_velocities):
    n_genes = true_velocities.shape[1]
    cosine_similarities = np.zeros(n_genes)

    for i in range(n_genes):
        true_v = true_velocities[:, i]
        est_v = estimated_velocities[:, i]

        true_v = true_v.reshape(1, -1)
        est_v = est_v.reshape(1, -1)

        cos_sim = cosine_similarity(true_v, est_v)[0][0]

        cosine_similarities[i] = cos_sim

    return cosine_similarities

In [38]:
compute_cosine_similarity(true_velocity, velocity).mean()

0.3711432920271955

In [40]:
compute_cosine_similarity(true_velocity_2, true_velocity).mean()

0.8176479923142969

In [38]:
adata= adata_dict['control']

In [39]:
   # Spliced cytoplasmic counts S_c = s_c(t)
gamma = adata.var['true_gamma'].values  # Degradation rates per gene
beta = adata.var['true_beta'].values
S = adata.layers['spliced']
U = adata.layers['unspliced']

In [54]:
true_velocity = beta*U - gamma*S

In [46]:
adata_dict_velocity

{'adata_s_u_800obs_300genes_deterministic': AnnData object with n_obs × n_vars = 800 × 300
     obs: 'true_t', 'n_counts', 'leiden', 'velocity_self_transition'
     var: 'true_t_', 'true_alpha', 'true_beta', 'true_nu', 'true_gamma', 'true_scaling', 'n_cells', 'velocity_gamma', 'velocity_qreg_ratio', 'velocity_r2', 'velocity_genes'
     uns: 'log1p', 'pca', 'neighbors', 'umap', 'leiden', 'velocity_params', 'velocity_graph', 'velocity_graph_neg'
     obsm: 'X_pca', 'X_umap'
     varm: 'PCs'
     layers: 'unspliced', 'spliced', 'spliced_cyt', 'Ms', 'Mu', 'velocity'
     obsp: 'distances', 'connectivities',
 'adata_n_c_800obs_300genes_deterministic': AnnData object with n_obs × n_vars = 800 × 300
     obs: 'true_t', 'n_counts', 'leiden', 'velocity_self_transition'
     var: 'true_t_', 'true_alpha', 'true_beta', 'true_nu', 'true_gamma', 'true_scaling', 'n_cells', 'velocity_gamma', 'velocity_qreg_ratio', 'velocity_r2', 'velocity_genes'
     uns: 'log1p', 'pca', 'neighbors', 'umap', 'leiden',

In [47]:
velocity = adata_dict_velocity['adata_control_deterministic'].layers['velocity']

In [55]:
compute_cosine_similarity(true_velocity,velocity).mean()

0.4882613733803661